# A Statistics Report Beside The Data

**Version**: 1

**Last modified**: 2026-10-12

**Compatible with opengate-gate-tree version:** 0.5.0

**Author**: Mateusz Jakub Bała

## About

Before anything is done with a file, it is worth knowing what it holds: how
many hits, how many events behind them, what range each branch covers, which
processes occur and how often. The package answers that with a report — one
summary that can be printed for a person and saved as JSON for whatever comes
next.

This notebook computes such a report on a file holding three runs, reads what
it says and where its numbers come from, saves it beside the data, and then
looks at the two cases where a report says more than it seems to: values that
stand for something, and a report computed on a selection rather than on the
file.

## What will you learn from this notebook

- how to compute a report on extracted data, print it, and save it as JSON
- why events are counted by the pair `(runID, eventID)` and tracks by the
  triple `(runID, eventID, trackID)`, and how the report says which it used
- how to read the numbers of the report from Python rather than from text
- how the values of a `PositroniumSource` appear in a report
- what values that are not finite do to a report
- why a report computed after a selection describes the selection, not the
  file

## Tutorial

Every step below is a line of text, the code that does what the line says, and
a word about the answer where the answer needs one.

In [1]:
from tools import environment, paths

print(environment.describe())

opengate-gate-tree: 0.5.0
python: 3.14.5
run on: 2026-10-12


The header of this notebook states the version it was written for; the line
above states the version it just ran against. If they differ, the notebook may
still be right — but nobody has checked it against what is installed.

Two example files: one whose hits GATE wrote in three trees, one per run,
and the positronium file of the earlier tutorials. Both are described in
[`../data/README.md`](../data/README.md).

In [2]:
output = paths.output_dir("tutorial-07-statistics-report")
per_run = paths.input_example("hits-one-tree-per-run.root")
positronium = paths.input_example("positronium-source-all-variants.root")

In [3]:
import json

import numpy as np

from opengate_gate_tree import (
    GammaType,
    GateTree,
    RootFile,
    TreeData,
    build_statistics_file_path,
    compute_statistics,
    format_statistics,
    statistics_to_dict,
    write_statistics,
)

### A Report On Three Runs

The three trees are read as one dataset, keeping the branches a first look
needs. The structure the first tree was recognised as is kept too: passed to the
report, it adds the variant and the tree name to it.

In [4]:
with RootFile(per_run) as root_file:
    print(root_file.hits_tree_names())
    detection = root_file.detect_hits_tree("Hits")
    hits = root_file.read_hits(
        branches=["runID", "eventID", "trackID", "edep", "time", "processName"]
    )

hits

('Hits', 'Hits_run1', 'Hits_run2')


TreeData(tree='Hits', entries=1500, branches=7)

Six branches were asked for and seven came back: reading several trees as
one adds `sourceTreeName`, which says where each row came from.

`compute_statistics` summarises the data, and `format_statistics` renders the
summary for reading:

In [5]:
statistics = compute_statistics(hits, detection)
print(format_statistics(statistics))

Tree: Hits
Hits tree variant: No system (A1), stored as 'Hits', 40 branches
Entries: 1500
Events: 458 (counted by runID, eventID)
Runs: 3
Tracks: 915 (counted by runID, eventID, trackID)
Deposited energy: 467.565
Time range: 1.59123e-06 to 0.0417158
Source trees: Hits, Hits_run1, Hits_run2
Branches (7):
  - runID / int32: min 0, max 2, mean 1, std 0.816497, 3 distinct
  - eventID / int32: min 0, max 153, mean 76.9013, std 44.3469, 154 distinct
  - trackID / int32: min 1, max 2, mean 1.49867, std 0.499998, 2 distinct
  - edep / float32: min 0.000601204, max 0.511, mean 0.31171, std 0.17063
  - time / float64: min 1.59123e-06, max 0.0417158, mean 0.020813, std 0.0163401
  - processName / text: 2 distinct, most frequent PhotoElectric (915), Compton (585)
  - sourceTreeName / text: 3 distinct, most frequent Hits (500), Hits_run1 (500), Hits_run2 (500)


The top part reads the hits as physics: 1500 hits from 458 events in 3
runs, 915 tracks, 467.6 MeV deposited in all — `edep` is in MeV, as GATE
writes it — and the times between which it happened. Below it, every branch
gets its range and spread; whole-number and text branches also get the number
of distinct values, and text branches the most frequent ones.

### What Counts As An Event

GATE numbers events within a run, starting again from zero in each. The
`eventID` branch on its own therefore holds far fewer distinct values than there
were events:

In [6]:
frame = hits.to_dataframe()

frame["eventID"].nunique(), len(frame[["runID", "eventID"]].drop_duplicates())

(154, 458)

154 distinct identifiers, but 458 distinct pairs of a run and an event — and
458 is what the report counted. Tracks are numbered within an event in the same
way, so they are counted by the triple. The report says which branches it used,
because that depends on what was extracted:

In [7]:
summary = statistics.hits_summary
summary.event_key, summary.event_count, summary.track_key, summary.track_count

(('runID', 'eventID'), 458, ('runID', 'eventID', 'trackID'), 915)

A selection without `runID` cannot count by the pair. The report then counts
by what is left, and says so — here the 154 identifiers, which in this file is
an undercount:

In [8]:
without_runs = compute_statistics(hits.select(["eventID", "trackID", "edep"]))
without_runs.hits_summary.event_key, without_runs.hits_summary.event_count

(('eventID',), 154)

The package leaves the identifiers as GATE wrote them and never renumbers
them, so an event is always named by the pair. Tutorial 11 is about working with
that in an analysis.

### Reading The Numbers From Python

The text above is for a person. Code reads the same numbers from the summary
itself: a `TreeStatistics` holding a `HitsSummary` and one `BranchStatistics`
per branch.

In [9]:
energy = next(branch for branch in statistics.branches if branch.name == "edep")
energy

BranchStatistics(name='edep', dtype='float32', kind=<BranchKind.FLOAT: 'float'>, entries=1500, non_finite_count=0, minimum=0.000601203937549144, maximum=0.5109999775886536, mean=0.311710000038147, std=0.17063041031360626, unique_count=None, top_values=())

In [10]:
processes = next(branch for branch in statistics.branches if branch.name == "processName")
processes.unique_count, processes.top_values

(2, (('PhotoElectric', 915), ('Compton', 585)))

### Saving The Report Beside The Data

`build_statistics_file_path` names the report the way the command line does,
`<title>.<tree>.stats.json`, so that it sits beside the data file of the same
title; `write_statistics` writes it.

In [11]:
report_path = write_statistics(
    statistics, build_statistics_file_path(output, "three-runs", GateTree.HITS)
)
report_path.name

'three-runs.hits.stats.json'

The file is JSON, because a report is meant to feed whatever comes next as
much as to be read: one run compared against another, a table of process
counts, a check that a file holds what a simulation was supposed to produce.
Its top part:

In [12]:
report = json.loads(report_path.read_text())

{key: report[key] for key in ["tree", "entries", "structure", "hits"]}

{'tree': 'Hits',
 'entries': 1500,
 'structure': {'variant': 'No system',
  'reference': 'A1',
  'branches': 40,
  'tree_name': 'Hits'},
 'hits': {'event_key': ['runID', 'eventID'],
  'events': 458,
  'runs': 3,
  'track_key': ['runID', 'eventID', 'trackID'],
  'tracks': 915,
  'total_edep': 467.56500244140625,
  'time_min': 1.5912298448870217e-06,
  'time_max': 0.041715848196351026,
  'source_trees': ['Hits', 'Hits_run1', 'Hits_run2']}}

and one branch:

In [13]:
report["branches"][3]

{'name': 'edep',
 'type': 'float32',
 'kind': 'float',
 'entries': 1500,
 'minimum': 0.000601203937549144,
 'maximum': 0.5109999775886536,
 'mean': 0.311710000038147,
 'std': 0.17063041031360626,
 'not_finite': 0}

`statistics_to_dict` gives the same content without writing a file, for code
that would rather keep it in memory:

In [14]:
statistics_to_dict(statistics) == report

True

### Values That Stand For Something

The three branches a `PositroniumSource` writes hold integers whose meaning the
package knows. A report names them rather than listing bare numbers, so it
says how many gammas were prompt rather than how many were `3`:

In [15]:
with RootFile(positronium) as root_file:
    positronium_detection = root_file.detect_hits_tree()
    positronium_hits = root_file.read(
        GateTree.HITS,
        ["runID", "eventID", "trackID", "edep", "sourceType", "decayType", "gammaType"],
    )

positronium_statistics = compute_statistics(positronium_hits, positronium_detection)
print(format_statistics(positronium_statistics))

Tree: Hits
Hits tree variant: No system (A1), stored as 'Hits', 40 branches
Entries: 500
Events: 115 (counted by runID, eventID)
Runs: 1
Tracks: 311 (counted by runID, eventID, trackID)
Deposited energy: 156.309
Branches (7):
  - runID / int32: min 0, max 0, mean 0, std 0, 1 distinct
  - eventID / int32: min 0, max 114, mean 58.116, std 32.9946, 115 distinct
  - trackID / int32: min 1, max 4, mean 1.844, std 0.882986, 4 distinct
  - edep / float32: min 0.000397417, max 1.3, mean 0.312618, std 0.22683
  - sourceType / int32: min 0, max 4, mean 2.456, std 1.08446, 4 distinct, most frequent ORTHO_POSITRONIUM (252), PARA_POSITRONIUM (138), NOT_DEFINED (61), DIRECT_ANNIHILATION (49)
  - decayType / int32: min 0, max 2, mean 1.238, std 0.652193, 3 distinct, most frequent STANDARD (259), DEEXCITATION (180), NONE (61)
  - gammaType / int32: min 0, max 3, mean 1.9, std 0.78867, 3 distinct, most frequent ANNIHILATION (367), PROMPT (72), UNKNOWN (61)


Among the 500 hits, 367 come from annihilation gammas, 72 from prompt ones
and 61 from gammas the source did not describe — those come from the second,
back-to-back source of the scene. The JSON report carries the names too:

In [16]:
positronium_report = statistics_to_dict(positronium_statistics)
next(branch for branch in positronium_report["branches"] if branch["name"] == "gammaType")

{'name': 'gammaType',
 'type': 'int32',
 'kind': 'integer',
 'entries': 500,
 'minimum': 0.0,
 'maximum': 3.0,
 'mean': 1.9,
 'std': 0.7886697661252141,
 'distinct_values': 3,
 'most_frequent': [{'value': 'ANNIHILATION', 'count': 367},
  {'value': 'PROMPT', 'count': 72},
  {'value': 'UNKNOWN', 'count': 61}]}

### Values That Are Not Finite

JSON can write neither "not a number" nor an infinity, and either would spoil
a mean. A report therefore counts such values and leaves them out of the
range, the mean and the deposited energy. GATE does not write them, but a
derived column can hold them — here the energy deposited per unit of track
length, made undefined wherever the length is zero:

In [17]:
energies = hits["edep"].astype(np.float64)
lengths = np.where(hits["trackID"] == 1, 0.0, 1.0)

with np.errstate(divide="ignore", invalid="ignore"):
    per_length = energies / lengths

derived = TreeData(GateTree.HITS, {"edep": energies, "perLength": per_length})
derived_branch = compute_statistics(derived).branches[1]
derived_branch.non_finite_count, derived_branch.minimum, derived_branch.maximum

(752, 0.002582278335466981, 0.5109999775886536)

The lengths here are made up — zero for every first track — only to produce
infinities in a real column. 752 of the 1500 values are not finite, one for
every hit of a first track; the range is taken over the other 748.

### A Report On A Selection Describes The Selection

A report summarises the data it is given. Computed after a selection, it
describes the selection — and nothing in its numbers says so. The prompt gammas
of the positronium file:

In [18]:
positronium_frame = positronium_hits.to_dataframe()
prompt = TreeData.from_dataframe(
    GateTree.HITS,
    positronium_frame[positronium_frame["gammaType"] == GammaType.PROMPT],
)

prompt_statistics = compute_statistics(prompt, positronium_detection)
print(format_statistics(prompt_statistics))

Tree: Hits
Hits tree variant: No system (A1), stored as 'Hits', 40 branches
Entries: 72
Events: 30 (counted by runID, eventID)
Runs: 1
Tracks: 30 (counted by runID, eventID, trackID)
Deposited energy: 39
Branches (7):
  - runID / int32: min 0, max 0, mean 0, std 0, 1 distinct
  - eventID / int32: min 4, max 112, mean 60.3472, std 32.7326, 30 distinct
  - trackID / int32: min 1, max 1, mean 1, std 0, 1 distinct
  - edep / float32: min 0.0100155, max 1.3, mean 0.541667, std 0.382177
  - sourceType / int32: min 2, max 3, mean 2.65278, std 0.476087, 2 distinct, most frequent ORTHO_POSITRONIUM (47), PARA_POSITRONIUM (25)
  - decayType / int32: min 2, max 2, mean 2, std 0, 1 distinct, most frequent DEEXCITATION (72)
  - gammaType / int32: min 3, max 3, mean 3, std 0, 1 distinct, most frequent PROMPT (72)


72 hits from 30 events, and every `gammaType` is `PROMPT` — true of the
selection, not of the file. The line describing the structure is the one place
that still speaks of the file: the recognised structure was passed along
unchanged, so it reports 40 branches while the report lists 7. A report saved
beside a filtered output should say in its name, or next to it, what the
filter was.

In [19]:
prompt_report = statistics_to_dict(prompt_statistics)
prompt_report["structure"]["branches"], len(prompt_report["branches"])

(40, 7)

## Summary

`compute_statistics` summarises extracted data — entries, events and tracks
counted by the identifiers that name them together, the range of every branch,
the named values of a `PositroniumSource` — `format_statistics` renders it for
reading, and `write_statistics` saves it as JSON beside the data. A report
describes exactly the data it was given, so one computed after a selection
describes the selection.

## Bibliography

- [Statistics](https://opengate-gate-tree.readthedocs.io/en/latest/guide/statistics.html)
  — what a report holds, from Python and from the command line
- [Event identifiers](https://opengate-gate-tree.readthedocs.io/en/latest/guide/events.html)
  — why an event is the pair `(runID, eventID)`
- [Merging trees](https://opengate-gate-tree.readthedocs.io/en/latest/guide/merging.html)
  — reading several trees as one dataset, and `sourceTreeName`
- [PositroniumSource Data](https://opengate-gate-tree.readthedocs.io/en/latest/guide/positronium.html)
  — the names a report gives to `sourceType`, `decayType` and `gammaType`
- [Units](https://opengate-gate-tree.readthedocs.io/en/latest/guide/units.html)
  — why the deposited energy is in MeV

## Changelog

- v1: written.